# Train regional model — Central Africa — Urban

Targets **PM2.5** (`pm25_surface`). Exports XGBoost JSON, LightGBM model, and `manifest.json` under `ml/exports/{region}/{segment}/`.

**Google Colab:** run the setup cell below first (optional: set secret `MFRAMAPA_CLONE_URL` if you use a fork). Skip that cell when running locally.

In [3]:
# Project setup: clone/find repo and install notebook dependencies
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

REPO_CLONE_URL = os.environ.get("MFRAMAPA_CLONE_URL", "https://github.com/yoadjei/Mframapa-AI.git")
IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
REQUIRED_MODULES = ("numpy", "pandas", "sklearn", "xgboost", "lightgbm", "shapely")


def _looks_like_repo(path: Path) -> bool:
    return (
        (path / "backend" / "data" / "african_cities.json").is_file()
        and (path / "ml").is_dir()
    )


def _missing_runtime_modules() -> list[str]:
    return [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]


def _candidate_roots() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates: list[Path] = [cwd, *cwd.parents]

    env_repo = os.environ.get("MFRAMAPA_REPO_DIR")
    if env_repo:
        env_path = Path(env_repo).expanduser()
        candidates.extend([env_path, *env_path.parents])

    candidates.extend(
        [
            Path("/content/Mframapa-AI"),
            Path("/content/Mframapa"),
            Path("/content/drive/MyDrive/Mframapa-AI"),
            Path("/content/drive/MyDrive/Mframapa"),
            Path.home() / "Mframapa-AI",
            Path.home() / "Mframapa",
        ]
    )

    seen: set[Path] = set()
    unique: list[Path] = []
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except FileNotFoundError:
            resolved = candidate.absolute()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def _find_repo_root() -> Path | None:
    return next((candidate for candidate in _candidate_roots() if _looks_like_repo(candidate)), None)


def _bootstrap_repo(*, install_deps: bool) -> Path | None:
    root = _find_repo_root()

    if root is None and IN_COLAB:
        repo_dir = Path(os.environ.get("MFRAMAPA_REPO_DIR", "/content/Mframapa-AI")).expanduser()
        if not _looks_like_repo(repo_dir):
            print(f"Cloning repo into {repo_dir}...")
            result = subprocess.run(
                ["git", "clone", REPO_CLONE_URL, str(repo_dir)],
                capture_output=True,
                text=True,
            )
            if result.returncode != 0 or not repo_dir.is_dir():
                if result.stderr:
                    print(result.stderr)
                raise RuntimeError(
                    "Git clone failed. If the repo is private, mount Google Drive and "
                    "upload the repo there, then set MFRAMAPA_REPO_DIR to its path."
                )
            print("Cloned successfully.")
        root = _find_repo_root() or (repo_dir if _looks_like_repo(repo_dir) else None)

    if root is None:
        return None

    os.environ["MFRAMAPA_REPO_DIR"] = str(root)
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    missing = _missing_runtime_modules()
    if install_deps and (IN_COLAB or missing):
        reason = (
            "Installing project dependencies for Colab runtime..."
            if IN_COLAB
            else f"Installing missing project dependencies for this kernel: {', '.join(missing)}"
        )
        print(reason)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
            check=True,
        )
        print("Dependencies ready.")
    elif missing:
        print(f"Kernel is missing packages: {', '.join(missing)}")
    else:
        print("Project dependencies already available in this kernel.")

    print(f"Repo root: {root}")
    return root

ROOT = _bootstrap_repo(install_deps=True)
if ROOT is None:
    print("Repo not auto-detected. Start this notebook from inside the repo, or set MFRAMAPA_REPO_DIR.")


Project dependencies already available in this kernel.
Repo root: /Users/davis/Documents/GitHub/Mframapa-AI


In [4]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

REPO_CLONE_URL = os.environ.get("MFRAMAPA_CLONE_URL", "https://github.com/yoadjei/Mframapa-AI.git")
IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
REQUIRED_MODULES = ("numpy", "pandas", "sklearn", "xgboost", "lightgbm", "shapely")


def _looks_like_repo(path: Path) -> bool:
    return (
        (path / "backend" / "data" / "african_cities.json").is_file()
        and (path / "ml").is_dir()
    )


def _missing_runtime_modules() -> list[str]:
    return [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]


def _candidate_roots() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates: list[Path] = [cwd, *cwd.parents]

    env_repo = os.environ.get("MFRAMAPA_REPO_DIR")
    if env_repo:
        env_path = Path(env_repo).expanduser()
        candidates.extend([env_path, *env_path.parents])

    candidates.extend(
        [
            Path("/content/Mframapa-AI"),
            Path("/content/Mframapa"),
            Path("/content/drive/MyDrive/Mframapa-AI"),
            Path("/content/drive/MyDrive/Mframapa"),
            Path.home() / "Mframapa-AI",
            Path.home() / "Mframapa",
        ]
    )

    seen: set[Path] = set()
    unique: list[Path] = []
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except FileNotFoundError:
            resolved = candidate.absolute()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def _find_repo_root() -> Path | None:
    return next((candidate for candidate in _candidate_roots() if _looks_like_repo(candidate)), None)


def _bootstrap_repo(*, install_deps: bool) -> Path | None:
    root = _find_repo_root()

    if root is None and IN_COLAB:
        repo_dir = Path(os.environ.get("MFRAMAPA_REPO_DIR", "/content/Mframapa-AI")).expanduser()
        if not _looks_like_repo(repo_dir):
            print(f"Cloning repo into {repo_dir}...")
            result = subprocess.run(
                ["git", "clone", REPO_CLONE_URL, str(repo_dir)],
                capture_output=True,
                text=True,
            )
            if result.returncode != 0 or not repo_dir.is_dir():
                if result.stderr:
                    print(result.stderr)
                raise RuntimeError(
                    "Git clone failed. If the repo is private, mount Google Drive and "
                    "upload the repo there, then set MFRAMAPA_REPO_DIR to its path."
                )
            print("Cloned successfully.")
        root = _find_repo_root() or (repo_dir if _looks_like_repo(repo_dir) else None)

    if root is None:
        return None

    os.environ["MFRAMAPA_REPO_DIR"] = str(root)
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    missing = _missing_runtime_modules()
    if install_deps and (IN_COLAB or missing):
        reason = (
            "Installing project dependencies for Colab runtime..."
            if IN_COLAB
            else f"Installing missing project dependencies for this kernel: {', '.join(missing)}"
        )
        print(reason)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
            check=True,
        )
        print("Dependencies ready.")
    elif missing:
        print(f"Kernel is missing packages: {', '.join(missing)}")
    else:
        print("Project dependencies already available in this kernel.")

    print(f"Repo root: {root}")
    return root

ROOT = _bootstrap_repo(install_deps=True)

if ROOT is None:
    searched = "\\n".join(f" - {candidate}" for candidate in _candidate_roots())
    raise RuntimeError(
        f"Cannot find repo root from {Path.cwd()}.\n"
        "Run the setup cell first in Colab, or set MFRAMAPA_REPO_DIR to the cloned repo.\n"
        f"Searched:\\n{searched}"
    )

REGION = "central_africa"
SEGMENT = "urban"

from ml.training import synthetic_training_frame, train_regional_bundle
from ml.model_selection import regional_export_dir

# Replace synthetic_training_frame(...) with your labelled parquet when available.
df = synthetic_training_frame(n_rows=1200, seed=42)
export_dir = regional_export_dir(REGION, SEGMENT)
result = train_regional_bundle(df, REGION, SEGMENT, export_dir, update_registry=True)
print(result)


Project dependencies already available in this kernel.
Repo root: /Users/davis/Documents/GitHub/Mframapa-AI
TrainingResult(r2_val_xgb=0.806846422280308, r2_val_lgb=0.8163853850082496, r2_val_ensemble=0.8225722588961839, conformal_half_width=10.848883059042578, export_dir=PosixPath('/Users/davis/Documents/GitHub/Mframapa-AI/ml/exports/central_africa/urban'))
